# Ridge Regression

**Project question:** Can L2 shrinkage improve prediction when many predictors are correlated?

By the end of this notebook, you should be able to:

- standardize predictors inside a leakage-safe pipeline
- tune ridge penalty strength with training-only cross-validation
- compare ridge with an OLS baseline on untouched test data

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

Ridge minimizes $\sum_i (y_i-\hat y_i)^2 + \alpha\sum_j \beta_j^2$. The intercept is not penalized by scikit-learn. Because the penalty depends on coefficient scale, predictors must be standardized inside the fitted workflow.

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv(DATA / 'simulated_correlated_predictors.csv')
X = df.drop(columns=['id', 'weekly_sales'])
y = df['weekly_sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=4031, test_size=0.30)
cv = KFold(n_splits=5, shuffle=True, random_state=4031)
pipe = make_pipeline(StandardScaler(), Ridge())
search = GridSearchCV(
    pipe, {'ridge__alpha': np.logspace(-3, 3, 25)},
    scoring='neg_root_mean_squared_error', cv=cv,
)
search.fit(X_train, y_train)
search.best_params_, -search.best_score_

The scaler is fit separately inside each training fold because it is part of the pipeline. The final test set has not influenced scaling, coefficient estimation, or alpha selection.

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

ols = LinearRegression().fit(X_train, y_train)
comparison = pd.DataFrame([
    {'model': 'OLS', 'test_rmse': rmse(y_test, ols.predict(X_test))},
    {'model': 'tuned ridge', 'test_rmse': rmse(y_test, search.predict(X_test))},
])
comparison

In [ ]:
ridge = search.best_estimator_.named_steps['ridge']
coef = pd.Series(
    ridge.coef_, index=X.columns, name='coefficient_per_1_sd_increase'
).sort_values(key=np.abs, ascending=False)
coef.head(12).to_frame()

**Interpretation:** Because predictors were standardized, these slopes are per one-training-standard-deviation increase and can be compared in magnitude. Ridge generally retains every predictor; small coefficients are shrunk, not selected away.